# CONFIRM (File 1b) — Preprocessing-Variant Inference on NHANES

The OAI experiment showed preprocessing order changes the grade on identical knees. But OAI and Mendeley are the
*same* knees, so a reviewer can say "maybe OAI is special." This repeats the test on **NHANES III** — genuinely
different patients, a different scanner, a different institution — to show the effect is **cross-institutional**,
not an OAI quirk.

NHANES's real pipeline (`nhanes3_batch_process.ipynb`) is **normalise(1–99 pct) → split L/R → CLAHE(2.0,8×8) →
resize 224**. The source films are public TIFFs on the CDC FTP, so this notebook re-downloads a sample and pushes
each knee through four pipeline orders with **one frozen model** — only the preprocessing differs.

| variant | pipeline |
|---|---|
| `nhanes_order` | normalise → split → **CLAHE → resize** (reproduces the real NHANES/OAI order — the reference) |
| `mrkr_order` | normalise → split → **resize → CLAHE** (the MRKR order) |
| `double_clahe` | `nhanes_order`, then CLAHE again at 224 |
| `no_clahe` | normalise → split → resize, no CLAHE (control) |

**Output:** `nhanes_preprocess_variants_findings.csv` → analysed by **File 2b** (`preprocess_crosscohort_analysis.ipynb`).
Uses the SAME frozen checkpoint as the OAI run so the two are directly comparable. No retraining.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib
sys.path.insert(0, '/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import numpy as np, pandas as pd, re, io, time, urllib.request, urllib.error
from pathlib import Path
import torch, torch.nn as nn
import cv2, tifffile
from PIL import Image
if 'training_lib_max' in sys.modules: importlib.reload(sys.modules['training_lib_max'])
import training_lib_max as TM
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda': raise RuntimeError('No GPU.')
PROJECT = Path('/content/drive/MyDrive/Master Thesis')
MT_ROOT = PROJECT/'scope3_mt'; MT_CKPT = MT_ROOT/'checkpoints'; MT_RES = MT_ROOT/'results'
mt_man = pd.read_csv(str(MT_ROOT/'manifest_mt.csv'))
SUB = [c for c in ['osteophyte_max','jsn_max','sclerosis_max'] if c in mt_man.columns]
print('sub-features:', SUB)

Mounted at /content/drive
sub-features: ['osteophyte_max', 'jsn_max', 'sclerosis_max']


In [ ]:
SUBK = 4
class MultiTaskNet(nn.Module):
    def __init__(self, n_sub):
        super().__init__()
        self.core = TM.OrdinalNet(config.NUM_CLASSES, 4, use_hierarchical=True)
        feat = self.core.feat_dim
        self.sub_heads = nn.ModuleList([nn.Sequential(nn.Flatten(1), nn.LayerNorm(feat), nn.Dropout(0.3),
                                                      nn.Linear(feat, SUBK-1)) for _ in range(n_sub)])
    def forward(self, x, grl_lambda=0.0):
        f = self.core.backbone(x)
        if f.dim() == 4: f = f.mean(dim=[-2,-1])
        kl = self.core.corn(f); s1 = self.core.head_s1(f); s2 = self.core.head_s2(f)
        dom = self.core.domain_head(TM.grad_reverse(f, grl_lambda))
        return kl, s1, s2, dom, [h(f) for h in self.sub_heads]
print('MultiTaskNet defined')

MultiTaskNet defined


In [ ]:
# ------------------------------- CONFIG -------------------------------
RUN_NAME = 'mt_mendeley_seed0'      # SAME frozen model as the OAI experiment (do NOT use mt_oai — no sub-labels)
N_KNEES  = 1200                     # sample of NHANES SEQNs to download+process (None = all with a label)
SEED     = 0
FTP_BASE = 'https://ftp.cdc.gov/pub/NHANES/XRays/Nhanes3/'
PNG_SUFFIX = 'norm-pct1-99_clahe-2.0_224px'
IMG_SIZE, CLAHE_CLIP, CLAHE_TILE = 224, 2.0, (8,8)
VARIANTS = ['nhanes_order','mrkr_order','double_clahe','no_clahe']
OUT_CSV = MT_RES/'nhanes_preprocess_variants_findings.csv'
print('variants:', VARIANTS)

variants: ['nhanes_order', 'mrkr_order', 'double_clahe', 'no_clahe']


In [ ]:
# NHANES labels + subject/side, from the manifest (dataset == nhanes3)
nh = mt_man[mt_man.dataset == 'nhanes3'].copy()
nh['seqn'] = nh.filename.astype(str).str.extract(r'SEQN0*(\d+)_')[0]
nh['side'] = nh.filename.astype(str).str.extract(r'SEQN\d+_([LR])_')[0]
nh = nh.dropna(subset=['seqn','side'])
print('NHANES knees with a label:', len(nh), '| unique SEQNs:', nh.seqn.nunique())
seqns = sorted(nh.seqn.unique(), key=int)
if N_KNEES:
    rng = np.random.default_rng(SEED)
    seqns = list(rng.choice(seqns, size=min(N_KNEES//2, len(seqns)), replace=False))
label_of = {(r.seqn, r.side): int(r.kl_grade) for r in nh.itertuples()}
print('SEQNs to fetch:', len(seqns))

NHANES knees with a label: 4785 | unique SEQNs: 2412
SEQNs to fetch: 600


In [ ]:
clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_TILE)

def normalise_film(arr):
    arr = arr.astype(np.float32)
    lo, hi = float(np.percentile(arr,1)), float(np.percentile(arr,99))
    if hi-lo < 1e-6: lo, hi = float(arr.min()), float(arr.max())
    if hi-lo < 1e-6: return np.zeros_like(arr, np.uint8)
    return np.clip((arr-lo)/(hi-lo)*255.0, 0, 255).astype(np.uint8)

def resize224(a): return np.array(Image.fromarray(a,'L').resize((IMG_SIZE,IMG_SIZE), Image.LANCZOS))

def make_variant(crop, v):
    if v == 'nhanes_order':  return resize224(clahe.apply(crop))            # CLAHE @ full-res -> resize (reference)
    if v == 'mrkr_order':    return clahe.apply(cv2.resize(crop,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LANCZOS4))
    if v == 'double_clahe':  return clahe.apply(resize224(clahe.apply(crop)))
    if v == 'no_clahe':      return resize224(crop)
    raise ValueError(v)

def fetch_tiff(seqn):
    url = f'{FTP_BASE}{int(seqn)}K1.tiff'
    req = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
    raw = tifffile.imread(io.BytesIO(urllib.request.urlopen(req, timeout=120).read()))
    return raw[...,0] if raw.ndim == 3 else raw
print('pipelines + fetch ready')

pipelines + fetch ready


In [ ]:
ckb = MT_CKPT/f'{RUN_NAME}_best.pt'
assert ckb.exists(), f'{ckb} not found'
model = MultiTaskNet(len(SUB)).to(device); TM.load_ckpt(str(ckb), model, None); model.eval()
print('reloaded', ckb)

@torch.no_grad()
def predict_one(img224):
    a = TM._resize(TM.joint_crop(img224))
    x = torch.from_numpy(a.astype(np.float32)/255.0); x = (x-0.485)/0.229
    xb = x.unsqueeze(0).repeat(3,1,1).unsqueeze(0).to(device)
    kl,_,_,_,subs = model(xb, grl_lambda=0.0)
    return int(TM.corn_probs(kl)[0].argmax()), [int(TM.corn_probs(s)[0].argmax()) for s in subs]

Downloading: "https://download.pytorch.org/models/convnext_large-ea097f82.pth" to /root/.cache/torch/hub/checkpoints/convnext_large-ea097f82.pth


100%|██████████| 755M/755M [00:03<00:00, 235MB/s]


reloaded /content/drive/MyDrive/Master Thesis/scope3_mt/checkpoints/mt_mendeley_seed0_best.pt


In [ ]:
rows=[]; t0=time.time(); n_err=0
for gi, seqn in enumerate(seqns):
    try:
        film = normalise_film(fetch_tiff(seqn))
    except Exception:
        n_err += 1; continue
    mid = film.shape[1]//2
    crops = {'R': film[:,:mid], 'L': np.fliplr(film[:,mid:])}
    for side, crop in crops.items():
        if (seqn, side) not in label_of: continue
        rec = {'seqn':seqn, 'side':side, 'knee_key':f'{seqn}{side}',
               'dataset':'nhanes3', 'kl_label':label_of[(seqn,side)]}
        for v in VARIANTS:
            try:
                kl, subp = predict_one(make_variant(crop, v))
                rec[f'{v}__kl_pred'] = kl
                for k,c in enumerate(SUB): rec[f'{v}__{c.replace("_max","")}_pred'] = subp[k]
            except Exception:
                rec[f'{v}__kl_pred'] = np.nan
        rows.append(rec)
    if (gi+1) % 100 == 0:
        el=time.time()-t0
        print('[%d/%d SEQNs] %d knees | %.0fs | ETA %.0fs | errs %d' %
              (gi+1, len(seqns), len(rows), el, el/(gi+1)*(len(seqns)-gi-1), n_err))

out = pd.DataFrame(rows)
out.to_csv(OUT_CSV, index=False)
print('\nsaved ->', OUT_CSV, '| knees:', len(out), '| download errors:', n_err)
print(out.head())

/tmp/ipykernel_1801/2148748449.py:10: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  def resize224(a): return np.array(Image.fromarray(a,'L').resize((IMG_SIZE,IMG_SIZE), Image.LANCZOS))


[100/600 SEQNs] 198 knees | 463s | ETA 2316s | errs 0
[200/600 SEQNs] 398 knees | 898s | ETA 1795s | errs 0
[300/600 SEQNs] 595 knees | 1365s | ETA 1365s | errs 0
[400/600 SEQNs] 794 knees | 1827s | ETA 914s | errs 0
[500/600 SEQNs] 992 knees | 2269s | ETA 454s | errs 0
[600/600 SEQNs] 1192 knees | 2719s | ETA 0s | errs 0

saved -> /content/drive/MyDrive/Master Thesis/scope3_mt/results/nhanes_preprocess_variants_findings.csv | knees: 1192 | download errors: 0
    seqn side knee_key  dataset  kl_label  nhanes_order__kl_pred  \
0  51097    R   51097R  nhanes3         2                      2   
1  51097    L   51097L  nhanes3         2                      2   
2  36578    R   36578R  nhanes3         0                      0   
3  36578    L   36578L  nhanes3         0                      0   
4  47635    R   47635R  nhanes3         0                      0   

   nhanes_order__osteophyte_pred  nhanes_order__jsn_pred  \
0                              1                       2   
1      